In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import gymnasium as gym
from copy import deepcopy

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [3]:
env_name = "HalfCheetah-v5"

In [4]:
def create_env(name=env_name, mode=None):
    return gym.make(name, render_mode=mode)

In [5]:
base_env = create_env()
action_scale = base_env.action_space.high[0]
print(f"Action scale: {action_scale}")
action_dim = base_env.action_space.shape[0]
state_dim = base_env.observation_space.shape[0]
print(action_dim, state_dim)

Action scale: 1.0
6 17


In [6]:
class TorchWrapper:

    def __init__(self, env, state_dim, action_dim, device=None):
        self.env = env
        self.state_dim = state_dim
        self.action_dim = action_dim
        self.device = device or torch.device("cpu")

    def np_to_torch(self, state):
        return torch.from_numpy(state).to(self.device)
    
    def reset(self):
        state, _ = self.env.reset()
        return torch.FloatTensor(state)
    
    def step(self, action):
        action = action.cpu().numpy()
        next_state, reward, terminated, truncated, _ = self.env.step(action)
        next_state = torch.FloatTensor(next_state)
        reward = torch.tensor(reward, dtype=torch.float).to(self.device)
        terminated = torch.tensor(terminated, dtype=torch.bool).to(self.device)
        truncated = torch.tensor(truncated, dtype=torch.bool).to(self.device)

        return next_state, reward, terminated, truncated


In [7]:
env = TorchWrapper(base_env, state_dim=state_dim, action_dim=action_dim)
env.reset()

tensor([-0.0229, -0.0962, -0.0572, -0.0049, -0.0297,  0.0613, -0.0051,  0.0951,
        -0.0480, -0.0770,  0.0099, -0.0657,  0.1000,  0.1441, -0.0097, -0.0317,
        -0.1235])

In [8]:
from collections import deque

class ReplayBuffer:

    def __init__(self, capacity=200_000, batch_size=128):
        self.capacity = capacity
        self.batch_size = batch_size
        self.replay_buffer = deque(maxlen=capacity)

    def store_transition(self, state, action, reward, next_state, done):
        self.replay_buffer.append([state, action, reward, next_state, done])

    def can_sample(self):
        return self.batch_size * 10 < len(self.replay_buffer)
    
    def sample(self):
        idxs = np.random.choice(len(self.replay_buffer), self.batch_size)
        states, actions, rewards, next_states, dones = [], [], [], [], []
        for idx in idxs:
            state, action, reward, next_state, done = self.replay_buffer[idx]
            states.append(state)
            actions.append(action)
            rewards.append(reward)
            next_states.append(next_state)
            dones.append(done)
        
        return torch.stack(states), torch.stack(actions), torch.stack(rewards).unsqueeze(1),\
              torch.stack(next_states), torch.stack(dones).unsqueeze(1)

In [9]:
class DQN(nn.Module):

    def __init__(self, state_dim, action_dim, hidden_size=400):
        super(DQN, self).__init__()
        self.dqn = nn.Sequential(
            nn.Linear(state_dim + action_dim, hidden_size),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_size, hidden_size),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_size, hidden_size),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_size, 1)
        )

    def forward(self, x):
        return self.dqn(x)

In [10]:
class PolicyNet(nn.Module):

    def __init__(self, state_dim, action_dim, action_scale, hidden_size=400):
        super(PolicyNet, self).__init__()
        self.action_scale = action_scale
        self.policy = nn.Sequential(
            nn.Linear(state_dim, hidden_size),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_size, hidden_size),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_size, hidden_size),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_size, action_dim),
            nn.Tanh()
        )

    def forward(self, x):
        return self.policy(x) * self.action_scale

In [15]:
from time import sleep


class DDPG:

    def __init__(self, env_name=env_name, capacity=200_000, batch_size=128, actor_lr=3e-4, critic_lr=3e-4, 
                 tau=0.001, gamma=0.99, num_iterations=10_000, epsilon_start=1., epsilon_end=0.2, epsilon_steps=500_000, device_arg=None):
        self.device = device_arg or torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.gamma = gamma
        self.epsilon = epsilon_start
        self.epsilon_end = epsilon_end
        self.eps_delta = (epsilon_start - epsilon_end) / epsilon_steps
        self.epsilon_steps = epsilon_steps
        self.tau = tau
        self.num_iterations = num_iterations

        self.env = TorchWrapper(create_env(env_name), state_dim, action_dim, device=self.device)

        self.replay_buffer = ReplayBuffer(capacity=capacity, batch_size=batch_size)

        self.q_network = DQN(state_dim=state_dim, action_dim=action_dim).to(self.device)
        self.policy_net = PolicyNet(state_dim=state_dim, action_dim=action_dim, action_scale=action_scale).to(self.device)

        self.target_q_network = deepcopy(self.q_network).eval()
        self.target_policy_net = deepcopy(self.policy_net).eval()

        self.actor_optim = optim.AdamW(self.policy_net.parameters(), lr=actor_lr)
        self.critic_optim = optim.AdamW(self.q_network.parameters(), lr=critic_lr)

    def load_models(self):
        print("Loading weignts...")
        self.q_network.load_state_dict(torch.load('./ddpg_q_network.pth'))
        self.target_q_network.load_state_dict(torch.load('./ddpg_q_network.pth'))

        self.policy_net.load_state_dict(torch.load('./ddpg_policy_network.pth'))
        self.target_policy_net.load_state_dict(torch.load('./ddpg_policy_network.pth'))
        print("Completed loading weights...")

    @torch.no_grad()
    # Add an eval flag to your policy method
    def policy(self, state, eval=False):
        state = state.to(self.device)
        action = self.policy_net(state)
        if not eval:
            action = action + torch.normal(0, self.epsilon, action.size(), device=self.device)
        return action.clamp(-action_scale, action_scale)
    
    def polyak_averaging(self, main_net, target_net):
        for mp, tp in zip(main_net.parameters(), target_net.parameters()):
            tp.data.copy_(self.tau * mp + (1 - self.tau) * tp)

    def update(self):
        states, actions, rewards, next_states, dones = self.replay_buffer.sample()
        states = states.to(self.device)
        actions = actions.to(self.device)
        rewards = rewards.to(self.device)
        next_states = next_states.to(self.device)
        dones = dones.to(self.device)
        not_dones = 1 - dones.float()
        with torch.no_grad():
            next_actions = self.target_policy_net(next_states)
            target_input = torch.cat((next_states, next_actions), dim=-1)
            y = rewards + self.gamma * not_dones * self.target_q_network(target_input)
        
        self.critic_optim.zero_grad()
        q_inputs = torch.cat((states, actions), dim=-1)
        q_values = self.q_network(q_inputs)
        
        critic_loss = F.mse_loss(q_values, y)
        critic_loss.backward()
        torch.nn.utils.clip_grad_norm_(self.q_network.parameters(), 1.0)
        self.critic_optim.step()

        for param in self.q_network.parameters():
            param.requires_grad = False


        self.actor_optim.zero_grad()
        policy_actions = self.policy_net(states)
        q_inputs = torch.cat((states, policy_actions), dim=-1)
        Q = self.q_network(q_inputs)
        actor_loss = - Q.mean()
        actor_loss.backward()
        torch.nn.utils.clip_grad_norm_(self.policy_net.parameters(), 1.0)
        self.actor_optim.step()

        for param in self.q_network.parameters():
            param.requires_grad = True

        self.polyak_averaging(self.q_network, self.target_q_network)
        self.polyak_averaging(self.policy_net, self.target_policy_net)
        self.update_epsilon()

    def update_epsilon(self):
        self.epsilon = max(self.epsilon_end, self.epsilon - self.eps_delta)

    def train(self):

        rewards = deque(maxlen=100)

        for i in range(self.num_iterations):

            state = self.env.reset()
            done = False
            ep_reward = 0.
            while not done:
                action = self.policy(state)
                next_state, reward, terminated, truncated = self.env.step(action)
                ep_reward += reward.item()
                done = terminated | truncated
                self.replay_buffer.store_transition(state, action, reward * 0.1, next_state, terminated)
                state = next_state

                if self.replay_buffer.can_sample():
                    self.update()

            rewards.append(ep_reward)
            print(f"Iteration: {i + 1}/{self.num_iterations} | ep reward: {ep_reward:.4}, Average reward: {np.mean(list(rewards)):.4} | epsilon: {self.epsilon:.4}")

    @torch.no_grad()
    def test(self, episodes=10):
        test_env = TorchWrapper(create_env(mode="human"), state_dim, action_dim, device=self.device)
        test_env.env.metadata["render_fps"] = 20

        for episode in range(1, 1 + episodes):
            state = test_env.reset()
            done = False
            ep_reward = 0.

            while not done:
                action = self.policy(state, True)
                next_state, reward, terminated, truncated = test_env.step(action)
                ep_reward += reward.item()
                done = terminated | truncated
                state = next_state

            print(f"Episode: {episode}, reward: {ep_reward}")
        test_env.env.close()


In [ ]:
ddpg = DDPG(device_arg=device, epsilon_start=1., epsilon_end=0.01)
ddpg.load_models()

Loading weignts...
Completed loading weights...


In [17]:
# ddpg.train()

In [20]:
ddpg.test()

Episode: 1, reward: 11096.098693311214
Episode: 2, reward: 7639.092260025442
Episode: 3, reward: 5375.494815645739
Episode: 4, reward: 10934.12720566988
Episode: 5, reward: 11207.930130124092
Episode: 6, reward: 10592.334554430097
Episode: 7, reward: 11774.042577907443
Episode: 8, reward: 11046.358339050785
Episode: 9, reward: 11466.44166982174
Episode: 10, reward: 10362.236843831837


In [ ]:
# torch.save(ddpg.q_network.state_dict(), 'ddpg_q_network.pth')
# torch.save(ddpg.policy_net.state_dict(), 'ddpg_policy_network.pth')